# Analisi visuale dei dataset di training

Questo notebook mostra le waveform dei due dataset per verificare se esistono
differenze visive sostanziali tra type1 e type2.

**Domanda:** se le waveform sono simili, nessun modello riuscirà a discriminare.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline

In [2]:
from src.data import load_waveforms

TYPE1_PATH = 'data/processed/Waveforms_type1_extended_128samples.txt'
TYPE2_PATH = 'data/processed/Waveforms_type2_extended_128samples.txt'

type1 = load_waveforms(TYPE1_PATH, expected_length=128)
type2 = load_waveforms(TYPE2_PATH, expected_length=128)

print(f'Type1:  {type1.shape[0]} waveforms × {type1.shape[1]} samples')
print(f'Type2:  {type2.shape[0]} waveforms × {type2.shape[1]} samples')
print()
print(f'Type1  stats: mean={type1.mean():.4f}, std={type1.std():.4f}, min={type1.min():.4f}, max={type1.max():.4f}')
print(f'Type2  stats: mean={type2.mean():.4f}, std={type2.std():.4f}, min={type2.min():.4f}, max={type2.max():.4f}')

FileNotFoundError: Waveform file not found: data/processed/Waveforms_type1_extended_128samples.txt

## 1. Waveform a caso (5 per classe)

Mostra 5 waveform scelte a caso da ciascun dataset, sovrapposte. 
Se sono tutte diverse tra loro e tra le classi, il clustering ha funzionato.
Se sono tutte uguali, il clustering ha raggruppato rumore.

In [ ]:
n_show = 5
rng = np.random.default_rng(42)

idx1 = rng.choice(type1.shape[0], n_show, replace=False)
idx2 = rng.choice(type2.shape[0], n_show, replace=False)

fig, axes = plt.subplots(2, 1, figsize=(14, 5))

# Type1
ax = axes[0]
for i, idx in enumerate(idx1):
    ax.plot(type1[idx], alpha=0.5, linewidth=0.8)
ax.set_title(f'Type1 — {n_show} waveform a caso (su {type1.shape[0]} totali)')
ax.set_xlabel('Sample index')
ax.set_ylabel('Amplitude')

# Type2
ax = axes[1]
for i, idx in enumerate(idx2):
    ax.plot(type2[idx], alpha=0.5, linewidth=0.8)
ax.set_title(f'Type2 — {n_show} waveform a caso (su {type2.shape[0]} totali)')
ax.set_xlabel('Sample index')
ax.set_ylabel('Amplitude')

plt.tight_layout()
plt.show()

## 2. Waveform stacked (media + std)

La waveform media è il pattern principale che il clustering ha identificato.
Le bande di std mostrano la variabilità interna.

**Guarda:** le due medie sono visivamente diverse o quasi identiche?

In [ ]:
mean1 = type1.mean(axis=0)
mean2 = type2.mean(axis=0)
std1 = type1.std(axis=0)
std2 = type2.std(axis=0)

fig, axes = plt.subplots(2, 1, figsize=(14, 6))

# Type1 stacked
ax = axes[0]
ax.plot(mean1, color='blue', linewidth=2, label='mean')
ax.fill_between(range(len(mean1)), mean1 - std1, mean1 + std1, color='blue', alpha=0.2)
ax.set_title(f'Type1 — Stacked (media ± std, {type1.shape[0]} waveforms)')
ax.set_xlabel('Sample index')
ax.set_ylabel('Amplitude')
ax.legend()
ax.grid(True, alpha=0.3)

# Type2 stacked
ax = axes[1]
ax.plot(mean2, color='orange', linewidth=2, label='mean')
ax.fill_between(range(len(mean2)), mean2 - std2, mean2 + std2, color='orange', alpha=0.2)
ax.set_title(f'Type2 — Stacked (media ± std, {type2.shape[0]} waveforms)')
ax.set_xlabel('Sample index')
ax.set_ylabel('Amplitude')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Superposizione diretta: Type1 vs Type2

Le due medie sullo stesso grafico. Questo è il test definitivo.
Se si sovrappongono → i due dataset sono praticamente identici → nessun modello li separerà.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(mean1, color='blue', linewidth=2.5, label=f'Type1 mean (n={type1.shape[0]})')
ax.fill_between(range(len(mean1)), mean1 - std1, mean1 + std1, color='blue', alpha=0.15)
ax.plot(mean2, color='orange', linewidth=2.5, label=f'Type2 mean (n={type2.shape[0]})')
ax.fill_between(range(len(mean2)), mean2 - std2, mean2 + std2, color='orange', alpha=0.15)

ax.set_title('Type1 vs Type2 — Waveform media sovrapposte', fontsize=14)
ax.set_xlabel('Sample index')
ax.set_ylabel('Amplitude')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Correlazione tra le medie

Coefficiente di correlazione Pearson tra le due medie. 
Se > 0.95 → le waveform sono quasi identiche.

In [ ]:
corr = np.corrcoef(mean1, mean2)[0, 1]
mse = np.mean((mean1 - mean2) ** 2)
max_diff = np.abs(mean1 - mean2).max()

print(f'Correlazione Pearson tra le medie: {corr:.4f}')
print(f'MSE tra le medie: {mse:.4f}')
print(f'Differenza massima assoluta: {max_diff:.4f}')
print(f'Correlazione media pairwise (1000 coppie): ', end='')

# Campiona 1000 waveform da ciascuna classe e calcola la correlazione media
n_pairs = 1000
idx1_sample = rng.choice(type1.shape[0], min(n_pairs, type1.shape[0]), replace=False)
idx2_sample = rng.choice(type2.shape[0], min(n_pairs, type2.shape[0]), replace=False)
n_pairs = min(n_pairs, len(idx1_sample), len(idx2_sample))
corrs = []
for i in range(n_pairs):
    c = np.corrcoef(type1[idx1_sample[i]], type2[idx2_sample[i]])[0, 1]
    corrs.append(c)
print(f'{np.mean(corrs):.4f} ± {np.std(corrs):.4f}')

## 5. Distribuzione delle ampiezze

Istogramma delle ampiezze (tutti i campioni di tutte le waveform) per ciascuna classe.
Se le distribuzioni sono diverse, le classi hanno caratteristiche statistiche diverse.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(type1.ravel(), bins=100, color='blue', alpha=0.7, edgecolor='black', linewidth=0.5)
axes[0].set_title(f'Type1 — Distribuzione ampiezze (n={type1.shape[0] * type1.shape[1]})')
axes[0].set_xlabel('Amplitude')
axes[0].set_ylabel('Count')
axes[0].axvline(type1.mean(), color='red', linestyle='--', label=f'mean={type1.mean():.2f}')
axes[0].legend()

axes[1].hist(type2.ravel(), bins=100, color='orange', alpha=0.7, edgecolor='black', linewidth=0.5)
axes[1].set_title(f'Type2 — Distribuzione ampiezze (n={type2.shape[0] * type2.shape[1]})')
axes[1].set_xlabel('Amplitude')
axes[1].set_ylabel('Count')
axes[1].axvline(type2.mean(), color='red', linestyle='--', label=f'mean={type2.mean():.2f}')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Stacked: tutte le waveform sovrapposte (con alpha bassa)

Qui mostriamo TUTTE le waveform del dataset sovrapposte con alpha bassa.
Se si vede una struttura chiara → le waveform sono coerenti.
Se si vede solo un blocco grigio → le waveform sono tutte diverse tra loro.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Type1: tutte le waveform
ax = axes[0]
n_sample = min(500, type1.shape[0])
sample_idx = rng.choice(type1.shape[0], n_sample, replace=False)
for idx in sample_idx:
    ax.plot(type1[idx], alpha=0.05, color='blue', linewidth=0.5)
ax.plot(mean1, color='blue', linewidth=2, label='mean', alpha=1.0)
ax.set_title(f'Type1 — Tutte le waveform sovrapposte (campione {n_sample}/{type1.shape[0]})')
ax.set_xlabel('Sample index')
ax.set_ylabel('Amplitude')
ax.legend()

# Type2: tutte le waveform
ax = axes[1]
n_sample = min(500, type2.shape[0])
sample_idx = rng.choice(type2.shape[0], n_sample, replace=False)
for idx in sample_idx:
    ax.plot(type2[idx], alpha=0.05, color='orange', linewidth=0.5)
ax.plot(mean2, color='orange', linewidth=2, label='mean', alpha=1.0)
ax.set_title(f'Type2 — Tutte le waveform sovrapposte (campione {n_sample}/{type2.shape[0]})')
ax.set_xlabel('Sample index')
ax.set_ylabel('Amplitude')
ax.legend()

plt.tight_layout()
plt.show()

## Verdetto

Guarda il plot di sovrapposizione diretta (sezione 3) e la correlazione (sezione 4).

- **Correlazione > 0.95**: le waveform sono praticamente identiche → il clustering non ha trovato pattern diversi
- **Correlazione 0.80-0.95**: le waveform sono simili ma con differenze misurabili → serve un modello più sensibile
- **Correlazione < 0.80**: le waveform sono diverse → il problema è il modello, non i dati